In [1]:
# ============================================
# UNSEEN DATA TESTING — All Three Datasets
# Evaluates each trained model against the true held-out test set
# ============================================

import pandas as pd
import numpy as np
import os
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC, LinearSVC
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, matthews_corrcoef, confusion_matrix)

def evaluate_unseen(X_train, y_train, X_test, y_test, models, pos_label, labels, dataset_name):
    """Train each model on full training data, test on held-out unseen test set."""
    results = []
    print(f"\n{'='*50}")
    print(f"UNSEEN DATA TESTING — {dataset_name}")
    print(f"{'='*50}")

    for name, model in models.items():
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred, pos_label=pos_label)
        rec = recall_score(y_test, y_pred, pos_label=pos_label)
        f1 = f1_score(y_test, y_pred, pos_label=pos_label)
        mcc = matthews_corrcoef(y_test, y_pred)
        tn, fp, fn, tp = confusion_matrix(y_test, y_pred, labels=labels).ravel()
        fpr = fp / (fp + tn)

        print(f"{name}: acc={acc:.4f}, prec={prec:.4f}, rec={rec:.4f}, f1={f1:.4f}, fpr={fpr:.4f}, mcc={mcc:.4f}")

        results.append({
            'model': name, 'accuracy': acc, 'precision': prec,
            'recall': rec, 'f1': f1, 'fpr': fpr, 'mcc': mcc
        })

    return pd.DataFrame(results).set_index('model')


# ==============================
# 1. NSL-KDD
# ==============================
train_df_nsl = pd.read_csv(r"C:\Users\icevo\OneDrive\Desktop\Dissertation\Datasets\NSL-KDD\processed\NSLKDD_train_cleaned.csv")
test_df_nsl = pd.read_csv(r"C:\Users\icevo\OneDrive\Desktop\Dissertation\Datasets\NSL-KDD\processed\NSLKDD_test_cleaned.csv")

X_train_nsl = train_df_nsl.drop(columns=['label', 'binary_label', 'difficulty'])
y_train_nsl = train_df_nsl['binary_label']
X_test_nsl = test_df_nsl.drop(columns=['label', 'binary_label', 'difficulty'])
y_test_nsl = test_df_nsl['binary_label']

models_nsl = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'KNN': KNeighborsClassifier(),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42),
    'SVM': SVC(random_state=42)
}

nsl_unseen_results = evaluate_unseen(
    X_train_nsl, y_train_nsl, X_test_nsl, y_test_nsl,
    models_nsl, pos_label='attack', labels=['normal', 'attack'],
    dataset_name='NSL-KDD'
)

# ==============================
# 2. UNSW-NB15
# ==============================
train_df_unsw = pd.read_csv(r"C:\Users\icevo\OneDrive\Desktop\Dissertation\Datasets\UNSW-NB15\processed\UNSWNB15_train_cleaned.csv")
test_df_unsw = pd.read_csv(r"C:\Users\icevo\OneDrive\Desktop\Dissertation\Datasets\UNSW-NB15\processed\UNSWNB15_test_cleaned.csv")

X_train_unsw = train_df_unsw.drop(columns=['label', 'attack_cat'])
y_train_unsw = train_df_unsw['label']
X_test_unsw = test_df_unsw.drop(columns=['label', 'attack_cat'])
y_test_unsw = test_df_unsw['label']

models_unsw = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'KNN': KNeighborsClassifier(),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42),
    'SVM': SVC(random_state=42)
}

unsw_unseen_results = evaluate_unseen(
    X_train_unsw, y_train_unsw, X_test_unsw, y_test_unsw,
    models_unsw, pos_label=1, labels=[0, 1],
    dataset_name='UNSW-NB15'
)

# ==============================
# 3. CICIDS2017
# ==============================
train_df_cic = pd.read_csv(r"C:\Users\icevo\OneDrive\Desktop\Dissertation\Datasets\CIC-IDS2017\processed\CICIDS2017_train_cleaned.csv")
test_df_cic = pd.read_csv(r"C:\Users\icevo\OneDrive\Desktop\Dissertation\Datasets\CIC-IDS2017\processed\CICIDS2017_test_cleaned.csv")

X_train_cic = train_df_cic.drop(columns=['Label', 'binary_label'])
y_train_cic = train_df_cic['binary_label']
X_test_cic = test_df_cic.drop(columns=['Label', 'binary_label'])
y_test_cic = test_df_cic['binary_label']

models_cic = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'KNN': KNeighborsClassifier(n_jobs=-1),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42, n_jobs=-1),
    'SVM': LinearSVC(random_state=42, max_iter=2000)
}

cic_unseen_results = evaluate_unseen(
    X_train_cic, y_train_cic, X_test_cic, y_test_cic,
    models_cic, pos_label='ATTACK', labels=['BENIGN', 'ATTACK'],
    dataset_name='CICIDS2017'
)

# ==============================
# 4. SAVE ALL RESULTS
# ==============================
results_folder = r"C:\Users\icevo\OneDrive\Desktop\Dissertation\Datasets\UnseenTesting_results"
os.makedirs(results_folder, exist_ok=True)

nsl_unseen_results.to_csv(os.path.join(results_folder, "UnseenTest_NSLKDD.csv"))
unsw_unseen_results.to_csv(os.path.join(results_folder, "UnseenTest_UNSWNB15.csv"))
cic_unseen_results.to_csv(os.path.join(results_folder, "UnseenTest_CICIDS2017.csv"))

print("\n\nAll unseen data testing results saved successfully!")
print(results_folder)


UNSEEN DATA TESTING — NSL-KDD
Logistic Regression: acc=0.7301, prec=0.9551, rec=0.5518, f1=0.6995, fpr=0.0343, mcc=0.5454
KNN: acc=0.6979, prec=0.8796, rec=0.5437, f1=0.6720, fpr=0.0983, mcc=0.4618
Decision Tree: acc=0.7252, prec=0.8965, rec=0.5849, f1=0.7079, fpr=0.0893, mcc=0.5079
Random Forest: acc=0.7495, prec=0.9634, rec=0.5821, f1=0.7257, fpr=0.0292, mcc=0.5763
SVM: acc=0.8063, prec=0.8622, rec=0.7852, f1=0.8219, fpr=0.1659, mcc=0.6138

UNSEEN DATA TESTING — UNSW-NB15
Logistic Regression: acc=0.7445, prec=0.6082, rec=0.9631, f1=0.7455, fpr=0.3944, mcc=0.5698
KNN: acc=0.7892, prec=0.6647, rec=0.9231, f1=0.7728, fpr=0.2960, mcc=0.6133
Decision Tree: acc=0.8237, prec=0.7090, rec=0.9268, f1=0.8034, fpr=0.2418, mcc=0.6678
Random Forest: acc=0.8307, prec=0.7044, rec=0.9726, f1=0.8170, fpr=0.2594, mcc=0.6971
SVM: acc=0.7528, prec=0.6125, rec=0.9906, f1=0.7569, fpr=0.3984, mcc=0.5974

UNSEEN DATA TESTING — CICIDS2017
Logistic Regression: acc=0.9571, prec=0.8809, rec=0.8627, f1=0.8717, f